# ⚽ Football-LLM — Serving Demo

Run the fine-tuned **Llama 3.1 8B** model with **vLLM** on a Colab **T4 GPU**.
This notebook launches the inference server, sends a test prediction, and starts an interactive **Gradio** demo with a public URL.

| Component | Details |
|---|---|
| Base model | `meta-llama/Llama-3.1-8B-Instruct` |
| LoRA adapter | `zanwenfu/football-llm-qlora` (83.9 MB) |
| Quantization | 4-bit (bitsandbytes NF4) |
| Max seq length | 768 tokens |

> **⚠️ Runtime** → Change runtime type → **T4 GPU** before running.

In [ ]:
# ── Cell 1: Install dependencies & authenticate ─────────────────────────────
!pip install -q vllm fastapi uvicorn gradio openai huggingface_hub

# Log in to HuggingFace (Llama 3.1 is a gated model)
# Option A: Colab Secrets — add a secret named HF_TOKEN in the sidebar
# Option B: Interactive prompt (paste your token when asked)
from huggingface_hub import login

try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
    print("✅ Logged in via Colab secret 'HF_TOKEN'")
except Exception:
    login()  # Interactive prompt

## 1 — Launch vLLM Server

Starts the **OpenAI-compatible** vLLM server as a background process.
The base model is loaded in **4-bit** (bitsandbytes NF4) with the LoRA adapter applied at runtime.
Model download takes **2-5 min** on first run (cached afterwards).

In [ ]:
# ── Cell 2: Launch vLLM server in the background ─────────────────────────────
import subprocess, sys, os

# Ensure the HF token is available to the subprocess (gated model)
env = os.environ.copy()
try:
    from google.colab import userdata
    env["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass  # Token already set by huggingface_hub.login() in Cell 1

# vLLM launch command — tuned for T4 (16 GB VRAM)
cmd = [
    sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--model",                 "meta-llama/Llama-3.1-8B-Instruct",
    "--enable-lora",
    "--lora-modules",          "football-llm=zanwenfu/football-llm-qlora",
    "--port",                  "8000",
    "--max-model-len",         "768",
    "--gpu-memory-utilization", "0.90",
    "--quantization",          "bitsandbytes",
    "--load-format",           "bitsandbytes",
    "--dtype",                 "float16",
    "--max-lora-rank",         "16",
    "--trust-remote-code",
]

log_path = "/tmp/vllm_server.log"
log_file = open(log_path, "w")

vllm_proc = subprocess.Popen(
    cmd,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env=env,
)

print(f"✅ vLLM server started (PID: {vllm_proc.pid})")
print(f"   Logs → {log_path}")
print(f"   Downloading model weights — this may take 2-5 min on first run...")

## 2 — Wait for Server

Polls `localhost:8000/health` every 10 s until the server is ready (typically 2-5 min on T4).
If it times out, check logs with `!cat /tmp/vllm_server.log`.

In [ ]:
# ── Cell 3: Wait for the vLLM server to be ready ─────────────────────────────
import time, requests

def wait_for_vllm(url="http://localhost:8000/health", timeout=600, interval=10):
    """Poll the vLLM health endpoint until it responds 200."""
    start = time.time()
    while time.time() - start < timeout:
        try:
            r = requests.get(url, timeout=5)
            if r.status_code == 200:
                elapsed = time.time() - start
                print(f"\n✅ vLLM server is ready!  ({elapsed:.0f}s)")
                return True
        except requests.ConnectionError:
            pass
        elapsed = time.time() - start
        print(f"⏳ Waiting for vLLM server... ({elapsed:.0f}s)")
        time.sleep(interval)
    raise TimeoutError(
        f"vLLM server did not start within {timeout}s.\n"
        f"Check logs: !cat /tmp/vllm_server.log"
    )

wait_for_vllm()

## 3 — Test Prediction

Sends a sample match (**Argentina vs France**, 2022 World Cup Final) using the same
compact prompt format used during training. The `openai` client points at the local vLLM server.

In [ ]:
# ── Cell 4: Send a test prediction request ───────────────────────────────────
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8000/v1", api_key="unused")

SYSTEM_PROMPT = (
    "You are a football match prediction model. "
    "Given team stats, predict the result, score, and brief reasoning."
)

# Same compact format as training data (generate_training_data.py)
user_message = """\
World Cup 2022 | Final | Lusail Stadium

Argentina (Home) | Coach: Lionel Scaloni | Formation: 4-3-3
Squad: 11 starters | Avg Rating: 7.2
Attack: 450 goals (0.35/90) | 180 assists | Top scorer: 200 goals
Defense: 120 yellows, 2 reds | Tackles/90: 0.60 | Duels: 55%
Passing: 72% accuracy

France (Away) | Coach: Didier Deschamps | Formation: 4-2-3-1
Squad: 11 starters | Avg Rating: 7.3
Attack: 420 goals (0.38/90) | 170 assists | Top scorer: 190 goals
Defense: 100 yellows, 1 reds | Tackles/90: 0.55 | Duels: 54%
Passing: 74% accuracy

Predict result, score, and reasoning."""

response = client.chat.completions.create(
    model="football-llm",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_message},
    ],
    max_tokens=200,
    temperature=0.1,
    top_p=0.9,
)

print("=" * 60)
print(f"  Model : {response.model}")
print(f"  Tokens: {response.usage.prompt_tokens} prompt + "
      f"{response.usage.completion_tokens} completion")
print("=" * 60)
print()
print(response.choices[0].message.content)

## 4 — Interactive Gradio Demo

Launches a **Gradio** interface with `share=True` to generate a **public URL** for demo.
Enter custom team stats or click one of the preset examples below the form.

In [ ]:
# ── Cell 5: Launch Gradio UI ─────────────────────────────────────────────────
import gradio as gr
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8000/v1", api_key="unused")

SYSTEM_PROMPT = (
    "You are a football match prediction model. "
    "Given team stats, predict the result, score, and brief reasoning."
)


def _fmt(val, decimals=1):
    """Format a number for the prompt, returning '-' for zero/missing."""
    if val is None or val == 0:
        return "-"
    if isinstance(val, float):
        return f"{val:.{decimals}f}"
    return str(int(val))


def predict_match(
    home_name, home_goals, home_gp90, home_assists, home_rating,
    home_top, home_yellows, home_reds, home_tackles, home_duels,
    home_pass, home_formation, home_coach,
    away_name, away_goals, away_gp90, away_assists, away_rating,
    away_top, away_yellows, away_reds, away_tackles, away_duels,
    away_pass, away_formation, away_coach,
    tournament, stage, venue,
):
    """Build the compact training-format prompt and call vLLM."""
    # Construct user message in the exact same format as training data
    user_msg = (
        f"{tournament} | {stage} | {venue}\n"
        f"\n"
        f"{home_name} (Home) | Coach: {home_coach} | Formation: {home_formation}\n"
        f"Squad: 11 starters | Avg Rating: {_fmt(float(home_rating))}\n"
        f"Attack: {_fmt(int(home_goals), 0)} goals ({_fmt(float(home_gp90), 2)}/90) | "
        f"{_fmt(int(home_assists), 0)} assists | Top scorer: {_fmt(int(home_top), 0)} goals\n"
        f"Defense: {_fmt(int(home_yellows), 0)} yellows, {_fmt(int(home_reds), 0)} reds | "
        f"Tackles/90: {_fmt(float(home_tackles), 2)} | Duels: {_fmt(float(home_duels), 0)}%\n"
        f"Passing: {_fmt(float(home_pass), 0)}% accuracy\n"
        f"\n"
        f"{away_name} (Away) | Coach: {away_coach} | Formation: {away_formation}\n"
        f"Squad: 11 starters | Avg Rating: {_fmt(float(away_rating))}\n"
        f"Attack: {_fmt(int(away_goals), 0)} goals ({_fmt(float(away_gp90), 2)}/90) | "
        f"{_fmt(int(away_assists), 0)} assists | Top scorer: {_fmt(int(away_top), 0)} goals\n"
        f"Defense: {_fmt(int(away_yellows), 0)} yellows, {_fmt(int(away_reds), 0)} reds | "
        f"Tackles/90: {_fmt(float(away_tackles), 2)} | Duels: {_fmt(float(away_duels), 0)}%\n"
        f"Passing: {_fmt(float(away_pass), 0)}% accuracy\n"
        f"\n"
        f"Predict result, score, and reasoning."
    )

    try:
        resp = client.chat.completions.create(
            model="football-llm",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": user_msg},
            ],
            max_tokens=200,
            temperature=0.1,
            top_p=0.9,
        )
        output = resp.choices[0].message.content.strip()

        # Parse structured output for a clean display
        lines = output.split("\n")
        pred_line  = next((l for l in lines if l.lower().startswith("prediction:")), "")
        score_line = next((l for l in lines if l.lower().startswith("score:")), "")
        reason_lines = []
        capture = False
        for l in lines:
            if l.lower().startswith("reasoning:"):
                reason_lines.append(l.split(":", 1)[1].strip())
                capture = True
            elif capture:
                reason_lines.append(l)
        reasoning = " ".join(reason_lines).strip() or output

        return (
            f"## ⚽ {home_name} vs {away_name}\n\n"
            f"**{pred_line}**\n\n"
            f"**{score_line}**\n\n"
            f"### Reasoning\n{reasoning}\n\n"
            f"---\n"
            f"<details><summary>Raw model output</summary>\n\n"
            f"```\n{output}\n```\n</details>"
        )
    except Exception as e:
        return f"## ❌ Error\n\n{str(e)}"


# --- Build the Gradio Interface ---
demo = gr.Interface(
    fn=predict_match,
    inputs=[
        # Home team (13 fields)
        gr.Textbox(label="🏠 Home Team",           value="Argentina"),
        gr.Number( label="Home Goals",              value=450),
        gr.Number( label="Home Goals/90",           value=0.35),
        gr.Number( label="Home Assists",            value=180),
        gr.Number( label="Home Avg Rating",         value=7.2),
        gr.Number( label="Home Top Scorer Goals",   value=200),
        gr.Number( label="Home Yellows",            value=120),
        gr.Number( label="Home Reds",               value=2),
        gr.Number( label="Home Tackles/90",         value=0.6),
        gr.Number( label="Home Duels %",            value=55),
        gr.Number( label="Home Pass Accuracy %",    value=72),
        gr.Textbox(label="Home Formation",          value="4-3-3"),
        gr.Textbox(label="Home Coach",              value="Lionel Scaloni"),
        # Away team (13 fields)
        gr.Textbox(label="✈️ Away Team",            value="France"),
        gr.Number( label="Away Goals",              value=420),
        gr.Number( label="Away Goals/90",           value=0.38),
        gr.Number( label="Away Assists",            value=170),
        gr.Number( label="Away Avg Rating",         value=7.3),
        gr.Number( label="Away Top Scorer Goals",   value=190),
        gr.Number( label="Away Yellows",            value=100),
        gr.Number( label="Away Reds",               value=1),
        gr.Number( label="Away Tackles/90",         value=0.55),
        gr.Number( label="Away Duels %",            value=54),
        gr.Number( label="Away Pass Accuracy %",    value=74),
        gr.Textbox(label="Away Formation",          value="4-2-3-1"),
        gr.Textbox(label="Away Coach",              value="Didier Deschamps"),
        # Match context (3 fields)
        gr.Textbox(label="🏆 Tournament",           value="World Cup 2022"),
        gr.Textbox(label="Stage",                   value="Final"),
        gr.Textbox(label="Venue",                   value="Lusail Stadium"),
    ],
    outputs=gr.Markdown(label="Prediction"),
    examples=[
        # Argentina vs France (2022 Final)
        ["Argentina", 450, 0.35, 180, 7.2, 200, 120, 2, 0.60, 55, 72, "4-3-3", "Lionel Scaloni",
         "France",    420, 0.38, 170, 7.3, 190, 100, 1, 0.55, 54, 74, "4-2-3-1", "Didier Deschamps",
         "World Cup 2022", "Final", "Lusail Stadium"],
        # Brazil vs Germany (2014 Semi Final)
        ["Brazil",  350, 0.30, 130, 6.9,  80, 140, 3, 0.50, 50, 68, "4-2-3-1", "Luiz Felipe Scolari",
         "Germany", 410, 0.36, 200, 7.1, 130, 110, 1, 0.58, 53, 75, "4-2-3-1", "Joachim Löw",
         "World Cup 2014", "Semi Final", "Estádio Mineirão"],
        # England vs Iran (2022 Group Stage)
        ["England", 308, 0.32, 140, 7.1, 109, 95, 1, 0.52, 52, 72, "4-3-3", "Gareth Southgate",
         "Iran",    139, 0.18,  60, 6.9,  77, 80, 2, 0.48, 49, 64, "4-4-2", "Carlos Queiroz",
         "World Cup 2022", "Group Stage", "Khalifa International Stadium"],
    ],
    title="⚽ Football-LLM — Match Predictor",
    description=(
        "Predict FIFA World Cup match outcomes using a fine-tuned Llama 3.1 8B model.\n"
        "Enter team statistics or click an example below to get started."
    ),
    allow_flagging="never",
)

demo.launch(share=True)